# Documentation of convert EU field boundaries to Pmtiles for web hosting services
We processed EU field boundaries from Planet into PMTiles to support Web GIS services such as EcodataCube(https://ecodatacube.eu).

- EU field boundaries data can be accessed via Zenodo: https://doi.org/10.5281/zenodo.14229033
- EU crop mapping on field boundaries via Zenodo: https://doi.org/10.5281/zenodo.14842659

In [1]:
# doownload crop boundaries from Zenodo
!wget -nc https://zenodo.org/records/14229033/files/field_boundaries.parquet?download=1
!wget -nc https://zenodo.org/records/14842659/files/field_boundaries_crop_classification.csv?download=1

File ‘field_boundaries.parquet?download=1’ already there; not retrieving.

File ‘field_boundaries_crop_classification.csv?download=1’ already there; not retrieving.



In [ ]:
from shapely.geometry import Polygon, Point
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
filed_bounaries = pd.read_parquet('field_boundaries.parquet?download=1')

/opt/conda/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.3-CAPI-1.17.3) is incompatible with the GEOS version PyGEOS was compiled with (3.11.1-CAPI-1.17.1). Conversions between both will be slow.
  warnings.warn(
/tmp/ipykernel_365/444943813.py:3: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling Py

In [5]:
df_crop=pd.read_csv('field_boundaries_crop_classification.csv?download=1')

In [7]:
filed_bounaries['id']=filed_bounaries['id'].astype(int)

In [8]:
filed_bounaries.head().set_index('id')

,area,geometry,determination_datetime,planet:ca_ratio,planet:micd,planet:qa,determination_method,bbox
id,,,,,,,,
35291833,0.270757,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0b\x00...",2022-06-01 00:00:00+00:00,0.391210,46.104290,0,auto-imagery,"{'xmin': 18.72719492257748, 'ymin': 45.8199928..."
35291842,0.627083,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\r\x00\x...,2022-06-01 00:00:00+00:00,0.553884,76.972801,0,auto-imagery,"{'xmin': 18.701706606680116, 'ymin': 45.819452..."
35291851,0.143134,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\n\x00\x...,2022-06-01 00:00:00+00:00,0.524641,33.431023,0,auto-imagery,"{'xmin': 18.701139716917233, 'ymin': 45.819363..."
35291869,10.217677,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1e\x00...",2022-06-01 00:00:00+00:00,2.750239,278.701172,0,auto-imagery,"{'xmin': 18.715886959814554, 'ymin': 45.819687..."
35291883,1.706494,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\r\x00\x...",2022-06-01 00:00:00+00:00,1.013447,97.955658,0,auto-imagery,"{'xmin': 18.695882937052108, 'ymin': 45.818939..."


In [9]:
df_join=filed_bounaries.set_index('id').join(df_crop.rename(columns={'field_boundary_id':'id'}).set_index('id'), on='id')

In [10]:
del filed_bounaries, df_crop

In [11]:
from shapely import from_wkb

In [12]:
df_join['geometry']=df_join.geometry.apply(lambda x: from_wkb(x))

In [13]:
gpd_filed_bounaries=df_join.set_geometry('geometry')

In [14]:
del df_join

In [15]:
df_join=None

In [16]:
gpd_filed_bounaries.to_parquet('field_boundaries_crop_classification.parquet')

In [17]:
gpd_filed_bounaries.head().columns

Index(['area', 'geometry', 'determination_datetime', 'planet:ca_ratio',
       'planet:micd', 'planet:qa', 'determination_method', 'bbox',
       'classification'],
      dtype='object')

In [18]:
gpd_filed_bounaries['area']=gpd_filed_bounaries['area'].astype('float32')

In [19]:
gpd_filed_bounaries.to_file('field_boundaries_crop_classification.geojson',Driver='GeoJSON')

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


In [20]:
# tippecanoe now support converting from GeoJSON
! tippecanoe  -o field_boundaries_crop_classification.pmtiles --force --maximum-tile-bytes=1000000 --no-feature-limit --read-parallel --processes=8 --hilbert --detect-shared-borders --coalesce-densest-as-needed --extend-zooms-if-still-dropping --full-detail=15  --low-detail=12 --minimum-zoom=0 --maximum-zoom=15 field_boundaries_crop_classification.geojson

tippecanoe: unrecognized option '--processes=8'
Usage: tippecanoe [options] [file.json ...]
  Output tileset
         --output=output.mbtiles [--output-to-directory=...] [--force]
         [--allow-existing]
  Tileset description and attribution
         [--name=...] [--attribution=...] [--description=...]
  Input files and layer names
         [--layer=...] [--named-layer=...]
  Parallel processing of input
         [--read-parallel]
  Projection of input
         [--projection=...]
  Zoom levels
         [--maximum-zoom=...] [--minimum-zoom=...]
         [--smallest-maximum-zoom-guess=...]
         [--extend-zooms-if-still-dropping]
         [--extend-zooms-if-still-dropping-maximum=...]
         [--generate-variable-depth-tile-pyramid] [--one-tile=...]
  Tile resolution
         [--full-detail=...] [--low-detail=...] [--minimum-detail=...]
         [--extra-detail=...]
  Filtering feature attributes
         [--exclude=...] [--include=...] [--exclude-all]
  Modifying feature attribu